> **SCAFFOLD — data pending.** The Llama-3.2-1B × OpenMathInstruct-2 runs do not exist yet: they require a Llama-tokenized OpenMath cache (`data/openmath_instruct_2_2m_packed_seq2048_llama32`) and the `*_llama32_1b_openmath_*` sweeps. Cells will error until those land. Group-naming convention and structure mirror `openmath_1b_leaderboard.ipynb`.

# Llama-3.2-1B × OpenMathInstruct-2 leaderboard — robustness pilot (9000 steps, ~225M slot-tokens)

Robustness pilot from `~/.claude/plans/as-part-of-our-tender-quilt.md`. Tests whether `adam-polar-product-lora-coupled-spectral-chord-tight` (plain k=1) keeps its eval-loss advantage over AdamW-LoRA when the dataset is swapped from opc-sft-stage2 (code-IFT) to OpenMathInstruct-2 (math-IFT, --column_map problem=instruction,generated_solution=output).

Cell: Llama-3.2-1B × OpenMathInstruct-2 train_2M (2M docs, ~200k packed slots @ seq=2048) × global_batch=16 (batch=4 × accum=4) × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000`, `eval_every=250`.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4}
- **chord-tight k=1** (`adam-polar-product-lora-coupled-spectral-chord-tight`): η ∈ {3e-3, 1e-2, 3e-2}

Source log groups: `{adamw,chord_tight}_robustness_llama32_1b_openmath_r{64,256}_blackwell` (4 groups total).

**σ anchor**: no per-dataset multi-seed AdamW run yet. Quoting Δ against `σ_AdamW(packed_v1, opc-sft-stage2, r=64) = 0.0017` as a **proxy only** — re-anchor before any paper claim.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure
from lora_playground.plotting.colors import OPTIM_COLORS, OPTIM_MARKERS

OPT_ADAMW = 'adamw'
OPT_CT    = 'adam-polar-product-lora-coupled-spectral-chord-tight'

def variant_key(cfg):
    opt = cfg.get('optimizer')
    if opt == OPT_ADAMW: return 'AdamW'
    if opt == OPT_CT:    return 'chord-tight k=1'
    return None

VARIANT_COLORS  = {'AdamW': OPTIM_COLORS.get(OPT_ADAMW, 'black'),
                   'chord-tight k=1': OPTIM_COLORS.get(OPT_CT, 'tab:red')}
VARIANT_MARKERS = {'AdamW': OPTIM_MARKERS.get(OPT_ADAMW, 'o'),
                   'chord-tight k=1': OPTIM_MARKERS.get(OPT_CT, 's')}

def render_cell(groups, rank, suptitle, final_ylim=None, traj_ylim=None):
    runs = load_runs(where={'log_group': groups}, logs_root='../logs',
                     warn_cross_commit=False, quiet=True)
    dedup = {}
    for cfg, hist in runs:
        k = (cfg['optimizer'], float(cfg['lr']))
        if k not in dedup or len(hist) > len(dedup[k][1]):
            dedup[k] = (cfg, hist)
    fig, table_df, summary_df = compare_variants_figure(
        variants={'AdamW': {'optimizer': OPT_ADAMW},
                  'chord-tight k=1': {'optimizer': OPT_CT}},
        common_where={'lora_r': rank}, ref_label='AdamW', sigma_ref=0.0017,
        max_steps=9000, allow_partial=True,
        prefetched_runs=list(dedup.values()), variant_key=variant_key,
        colors=VARIANT_COLORS, markers=VARIANT_MARKERS,
        suptitle=suptitle, figsize=(11, 4),
        final_ylim=final_ylim, traj_ylim=traj_ylim,
    )
    plt.show()
    print(f'--- {suptitle} per-η table ---')
    display(table_df.style.format('{:.4f}', na_rep='—'))
    print(f'--- {suptitle} summary ---')
    display(summary_df.style.format({
        'best_lr': '{:.1e}', 'final': '{:.4f}',
        'delta': '{:+.4f}', 'delta_sigma': '{:+.2f}σ'
    }, na_rep='—'))

## r=64

In [ ]:
GROUPS_R64 = [
    'adamw_robustness_llama32_1b_openmath_r64_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r64_blackwell',
    'adamw_robustness_llama32_1b_openmath_r64_ext_right_blackwell',   # right-extension (adamw r64 was pinned at 3e-4)
    'chord_tight_robustness_llama32_1b_openmath_r64_ext_right_blackwell',  # right-extension (chord r64 pinned at 3e-2)
]
render_cell(GROUPS_R64, 64, 'OpenMathInstruct-2 r=64',
            final_ylim=(0.39, 0.55), traj_ylim=(0.39, 0.55))

## r=256

In [ ]:
GROUPS_R256 = [
    'adamw_robustness_llama32_1b_openmath_r256_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r256_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r256_ext_right_blackwell',  # right-extension (chord r256 pinned at 3e-2)
]
render_cell(GROUPS_R256, 256, 'OpenMathInstruct-2 r=256')

## r=64 — NS-iteration ablation (ns=5 vs ns=8)

Does ns=8 (full Newton–Schulz whitening) beat the ns=5 base on math-IFT, as it did on Llama×code? AdamW + chord ns=5 are the existing gpuxl runs; chord ns=8 is the new Blackwell arm. Hardware mix is flagged in-cell — loss ranking is hardware-independent.

In [ ]:
# NS-iteration ablation: does ns=8 (full whitening) beat ns=5 on math-IFT?
# HARDWARE NOTE: ns=8 ran on Blackwell; AdamW + ns=5 on gpuxl (h200/h100).
# Loss ranking is hardware-independent; the mix is flagged for transparency.
GROUPS_NS_R64 = [
    'adamw_robustness_llama32_1b_openmath_r64_blackwell',
    'adamw_robustness_llama32_1b_openmath_r64_ext_right_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r64_blackwell',              # ns=5 (base)
    'chord_tight_robustness_llama32_1b_openmath_r64_ext_right_blackwell',    # ns=5 (lr extension)
    'chord_tight_robustness_llama32_1b_openmath_r64_ns8_blackwell',      # ns=8
]
def ns_of(cfg):
    oc = cfg.get('optimizer_config') or {}
    return cfg.get('muon_ns_steps', oc.get('ns_steps'))
def variant_key_ns(cfg):
    opt = cfg.get('optimizer')
    if opt == OPT_ADAMW: return 'AdamW'
    if opt == OPT_CT:
        ns = ns_of(cfg)
        return f'chord ns={ns}' if ns is not None else 'chord ns=?'
    return None
runs = load_runs(where={'log_group': GROUPS_NS_R64}, logs_root='../logs',
                 warn_cross_commit=False, quiet=True)
dedup = {}
for cfg, hist in runs:
    k = (cfg['optimizer'], float(cfg['lr']), ns_of(cfg))
    if k not in dedup or len(hist) > len(dedup[k][1]):
        dedup[k] = (cfg, hist)
NS_COLORS = {'AdamW': OPTIM_COLORS.get(OPT_ADAMW, 'black'),
             'chord ns=5': '#ff7f0e', 'chord ns=8': '#2ca02c'}
NS_MARKERS = {'AdamW': 'o', 'chord ns=5': 's', 'chord ns=8': '^'}
fig, table_df_ns, summary_df_ns = compare_variants_figure(
    variants={'AdamW': {}, 'chord ns=5': {}, 'chord ns=8': {}},
    common_where={'lora_r': 64}, ref_label='AdamW', sigma_ref=0.0017,
    max_steps=9000, allow_partial=True,
    prefetched_runs=list(dedup.values()), variant_key=variant_key_ns,
    colors=NS_COLORS, markers=NS_MARKERS,
    suptitle='OpenMathInstruct-2 r=64 — NS-iteration (whitening): ns=5 vs ns=8',
    figsize=(11, 4), final_ylim=(0.39, 0.55), traj_ylim=(0.39, 0.55),
)
plt.show()
print('--- r=64 NS per-η table ---')
display(table_df_ns.style.format('{:.4f}', na_rep='—'))
print('--- r=64 NS summary ---')
display(summary_df_ns.style.format({
    'best_lr': '{:.1e}', 'final': '{:.4f}',
    'delta': '{:+.4f}', 'delta_sigma': '{:+.2f}σ'
}, na_rep='—'))

## r=256 — NS-iteration ablation (ns=5 vs ns=8)

In [ ]:
# r=256 NS-iteration (reuses ns_of / variant_key_ns from the r=64 NS cell above).
# HARDWARE NOTE: ns=8 on Blackwell; AdamW + ns=5 on gpuxl.
GROUPS_NS_R256 = [
    'adamw_robustness_llama32_1b_openmath_r256_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r256_blackwell',            # ns=5 (base)
    'chord_tight_robustness_llama32_1b_openmath_r256_ext_right_blackwell',  # ns=5 (lr extension)
    'chord_tight_robustness_llama32_1b_openmath_r256_ns8_blackwell',    # ns=8
]
runs = load_runs(where={'log_group': GROUPS_NS_R256}, logs_root='../logs',
                 warn_cross_commit=False, quiet=True)
dedup = {}
for cfg, hist in runs:
    k = (cfg['optimizer'], float(cfg['lr']), ns_of(cfg))
    if k not in dedup or len(hist) > len(dedup[k][1]):
        dedup[k] = (cfg, hist)
NS_COLORS_R256 = {'AdamW': OPTIM_COLORS.get(OPT_ADAMW, 'black'),
                  'chord ns=5': '#ff7f0e', 'chord ns=8': '#2ca02c'}
NS_MARKERS_R256 = {'AdamW': 'o', 'chord ns=5': 's', 'chord ns=8': '^'}
fig, table_df_ns_r256, summary_df_ns_r256 = compare_variants_figure(
    variants={'AdamW': {}, 'chord ns=5': {}, 'chord ns=8': {}},
    common_where={'lora_r': 256}, ref_label='AdamW', sigma_ref=0.0017,
    max_steps=9000, allow_partial=True,
    prefetched_runs=list(dedup.values()), variant_key=variant_key_ns,
    colors=NS_COLORS_R256, markers=NS_MARKERS_R256,
    suptitle='OpenMathInstruct-2 r=256 — NS-iteration (whitening): ns=5 vs ns=8',
    figsize=(11, 4), final_ylim=(0.375, 0.46), traj_ylim=(0.375, 0.51),
)
plt.show()
print('--- r=256 NS per-η table ---')
display(table_df_ns_r256.style.format('{:.4f}', na_rep='—'))
print('--- r=256 NS summary ---')
display(summary_df_ns_r256.style.format({
    'best_lr': '{:.1e}', 'final': '{:.4f}',
    'delta': '{:+.4f}', 'delta_sigma': '{:+.2f}σ'
}, na_rep='—'))